## 1. Import Libraries

We install our dependencies and import libraries required for our data quality engine.

In [ ]:
# Install required libraries
%pip install pandas numpy openpyxl matplotlib jinja2

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

## 2. Project Paths

We dynamically determine the project root folder regardless of where the notebook was launched, and establish paths to our raw data and reports.

In [ ]:
current_dir = Path.cwd()

# Dynamically resolve the project root
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

RAW_DATA = PROJECT_ROOT / "data" / "raw"
REPORT_PATH = PROJECT_ROOT / "reports"

# Ensure the reports directory exists
REPORT_PATH.mkdir(exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Data:", RAW_DATA)

## 3. Find All Excel Files

We locate and list all the Excel datasets in our raw data directory that will be scanned for quality.

In [ ]:
excel_files = sorted(RAW_DATA.glob("*.xlsx"))

print(f"Datasets Found: {len(excel_files)}")

for file in excel_files:
    print(file.name)

## 4. Data Quality Analysis

We scan every dataset, calculating total rows, columns, missing values, duplicates, and memory usage to compile a comprehensive data quality report.

In [ ]:
quality_report = []

for file in excel_files:
    try:
        df = pd.read_excel(file)

        quality_report.append({
            "Dataset": file.name,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Missing Values": int(df.isnull().sum().sum()),
            "Duplicate Rows": int(df.duplicated().sum()),
            "Memory Usage (KB)": round(df.memory_usage(deep=True).sum()/1024, 2)
        })
    except Exception as e:
        print(f"Error reading {file.name}: {e}")

quality_df = pd.DataFrame(quality_report)
quality_df

## 5. Missing Values by Dataset

We break down the missing values for each column across all datasets.

In [ ]:
for file in excel_files:
    try:
        print("=" * 70)
        print(file.name)
        df = pd.read_excel(file)
        display(df.isnull().sum())
    except:
        pass

## 6. Duplicate Rows

We extract exactly how many rows are completely duplicated in each file.

In [ ]:
duplicates = []

for file in excel_files:
    try:
        df = pd.read_excel(file)
        duplicates.append({
            "Dataset": file.name,
            "Duplicate Rows": int(df.duplicated().sum())
        })
    except:
        pass

duplicates_df = pd.DataFrame(duplicates)
duplicates_df

## 7. Data Types

We inspect the Pandas-inferred data types for all columns across every dataset to ensure consistency.

In [ ]:
for file in excel_files:
    try:
        print("=" * 60)
        print(file.name)
        df = pd.read_excel(file)
        display(df.dtypes)
    except:
        pass

## 8. Dataset Health Score

We calculate a custom 'Health Score' starting at 100, deducting points for missing values and duplicates to give each dataset a quality grade.

In [ ]:
if not quality_df.empty:
    quality_df["Health Score"] = 100
    quality_df["Health Score"] -= quality_df["Missing Values"] * 0.05
    quality_df["Health Score"] -= quality_df["Duplicate Rows"] * 2
    quality_df["Health Score"] = quality_df["Health Score"].clip(lower=0)

quality_df

## 9. Save Report

We export the finalized Data Quality Report as a CSV file to our reports folder for project documentation.

In [ ]:
if not quality_df.empty:
    quality_df.to_csv(
        REPORT_PATH / "data_quality_report.csv",
        index=False
    )
    print("Data Quality Report Saved Successfully")
else:
    print("No data to save.")

## 10. Summary

We aggregate the metrics into a final grand total to understand the overall health and scope of our data ecosystem.

In [ ]:
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

if not quality_df.empty:
    print("Datasets:", len(quality_df))
    print("Total Rows:", quality_df["Rows"].sum())
    print("Total Missing:", quality_df["Missing Values"].sum())
    print("Total Duplicates:", quality_df["Duplicate Rows"].sum())
    display(quality_df)
else:
    print("No data available to summarize.")